# PROJ951 · TD1 · The bill and the energy balance

<table>
  <tr>
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/PROJ951/main/notebooks/logos/solaracademy.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/PROJ951/main/notebooks/logos/usmb.png" width="200" />
    <td><img src="https://raw.githubusercontent.com/Mohamed-ENNHIRI/PROJ951/main/notebooks/logos/cnrs.png" width="80" />
  </tr>
</table>

<sub>Author: Mohamed Ennhiri, PhD student, LOCIE UMR CNRS 5271, Université Savoie Mont Blanc</sub>

<sub>Data: PVGIS 5.3 (SARAH3 and ERA5) for the production, synthetic profiles for the demand of the three buildings. Every price, tariff and carbon factor is a course value, set for teaching.</sub>

**Goal.** A first money estimate for the school, with monthly totals: the value of one self-consumed kWh, the monthly energy balance, the year-one money of scenario A (total sale) and scenario B (self-consumption with surplus sale), and a simple payback for each.

**Case.** A school near Chambéry, 99 kWp on the roof, 140 MWh of demand per year. Every group works on the school today. TP1 redoes the work for your own building.

**Time.** 1 h 30: part 1, 15 min. Part 2, 30 min. Part 3, 25 min. Part 4, 10 min.

**How to work.** Work in pairs, one ESBC and one SoLEM student. Each task asks for one line of code: replace every `...` and every `# TODO`, then run the check under the task. Write your answers in the cells marked **Answer**. First action on Colab: **File, Save a copy in Drive**.

Names:

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

In [ ]:
url = "https://raw.githubusercontent.com/Mohamed-ENNHIRI/PROJ951/main/data/proj951_case_data.csv"
df = pd.read_csv(url, index_col="time", parse_dates=True)
print(df.shape)
df.head()

In [ ]:
SIZE_KWP = 99                   # kWp
SUPPLY = 0.11                   # EUR/kWh, energy supply, variable part of the bill
NETWORK = 0.04                  # EUR/kWh, network tariff (TURPE), variable part
TAXES = 0.03                    # EUR/kWh, taxes per kWh (accise and others)
TARIFF_TOTAL_SALE = 0.10        # EUR/kWh, scenario A
TARIFF_SURPLUS = 0.07           # EUR/kWh, scenario B, surplus
CAPEX_EUR_PER_WP = 1.10         # investment, EUR per Wp
OPEX_EUR_PER_KWP_YEAR = 20.0    # operating cost, EUR per kWp and year

pv = df["pv_kwh_per_kwp"] * SIZE_KWP     # kWh per hour, production of the roof
load = df["load_school_kwh"]             # kWh per hour, demand of the school
print(f"Production {pv.sum() / 1000:.1f} MWh, demand {load.sum() / 1000:.1f} MWh")

**Course values.** Every price in this notebook is set for teaching. None of them is a tariff in force. TD3 looks up the values in force.

## Part 1. The electricity bill (15 min)

The bill has a fixed part, paid each year whatever the consumption, and a variable part, paid per kWh. A kWh produced and used on site replaces a kWh bought from the grid, so only the variable part falls. VAT stays out of the course: a professional buyer recovers it.

### Task 1.1. Value of one self-consumed kWh

Compute the grid price avoided by one self-consumed kWh (sum of the three variable lines), the share of this price going to supply, the variable part of the school bill without PV, and the ratio between the grid price and the surplus tariff.

In [ ]:
grid_price = SUPPLY + NETWORK + TAXES
share_supply = SUPPLY / grid_price
variable_bill = load.sum() * grid_price
ratio = grid_price / TARIFF_SURPLUS

print(f"Grid price avoided:          {grid_price:.2f} EUR/kWh")
print(f"Share of supply:             {100 * share_supply:.0f} %")
print(f"Variable bill without PV:    {variable_bill:,.0f} EUR")
print(f"Self-consumed kWh / surplus: {ratio:.1f}")

**Check.** 0.18 EUR/kWh, 61 %, about 25,200 EUR, 2.6.

**Question 1.** Why does the fixed part of the bill stay the same after the PV system starts?

**Question 2.** What does the ratio of task 1.1 tell the owner about the best use of each kWh?

**Answer.**

**Solution.** Q1: the fixed part pays for the connection and the subscribed power. The school stays connected and still needs full power from the grid in the evening, in winter and on cloudy days. Q2: a kWh used on site is worth 2.6 times a kWh sold as surplus. Self-consumption, not production, drives the value of scenario B.

## Part 2. The monthly energy balance (30 min)

Each month, the school uses the PV production first:
- self-consumed energy = the smaller of production and demand,
- surplus = production minus self-consumed energy, sold to the grid,
- import = demand minus self-consumed energy, bought from the grid.

### Task 2.1. Monthly table

The first line sums the hours of each month, in MWh. Complete the three columns. `np.minimum(a, b)` gives the smaller of two values, row by row.

In [ ]:
monthly = pd.DataFrame({"pv": pv, "load": load}).resample("MS").sum() / 1000
monthly.index = monthly.index.strftime("%b")

monthly["self_consumed"] = np.minimum(monthly["pv"], monthly["load"])
monthly["surplus"] = monthly["pv"] - monthly["self_consumed"]
monthly["import"] = monthly["load"] - monthly["self_consumed"]
monthly.round(1)

### Task 2.2. Year totals and two rates

- Self-consumption rate = self-consumed energy / production: the share of the production used on site.
- Self-sufficiency rate = self-consumed energy / demand: the share of the demand covered by PV.

In [ ]:
self_consumed_mwh = monthly["self_consumed"].sum()
surplus_mwh = monthly["surplus"].sum()
sc_rate = self_consumed_mwh / monthly["pv"].sum()
ss_rate = self_consumed_mwh / monthly["load"].sum()

print(f"Self-consumed: {self_consumed_mwh:.1f} MWh, surplus: {surplus_mwh:.1f} MWh")
print(f"Self-consumption rate: {100 * sc_rate:.1f} %, self-sufficiency rate: {100 * ss_rate:.1f} %")

monthly[["self_consumed", "surplus"]].plot(kind="bar", stacked=True, color=["seagreen", "orange"])
plt.plot(range(12), monthly["load"], "k_", markersize=20, label="demand")
plt.ylabel("Energy per month (MWh)")
plt.legend()
plt.show()

**Check.** Self-consumed plus surplus gives the production. Self-consumed energy near 107 MWh.

**Question 3.** Which months show a surplus? Explain with the school calendar.

**Answer.**

**Solution.** April, June, July and August. July and August fall in the summer holidays: the school stands nearly empty while production peaks. April holds the spring holidays. June combines the longest days with a demand falling at the end of the school year.

## Part 3. Year-one money (25 min)

- Scenario A: the whole production sold at the total-sale tariff.
- Scenario B: the self-consumed kWh avoid the grid price, the surplus sells at the surplus tariff.
- Simple payback = CAPEX / (year-one revenue − year-one OPEX).

### Task 3.1. Revenue and simple payback

Watch the units: the price holds per Wp, the size in kWp. The energy of part 2 holds in MWh, the prices in EUR/kWh.

In [ ]:
capex = CAPEX_EUR_PER_WP * SIZE_KWP * 1000
opex = OPEX_EUR_PER_KWP_YEAR * SIZE_KWP
revenue_a = pv.sum() * TARIFF_TOTAL_SALE
revenue_b = self_consumed_mwh * 1000 * grid_price + surplus_mwh * 1000 * TARIFF_SURPLUS
payback_a = capex / (revenue_a - opex)
payback_b = capex / (revenue_b - opex)

print(f"CAPEX {capex:,.0f} EUR, OPEX {opex:,.0f} EUR per year")
print(f"A: revenue {revenue_a:,.0f} EUR, simple payback {payback_a:.1f} years")
print(f"B: revenue {revenue_b:,.0f} EUR, simple payback {payback_b:.1f} years")

**Check.** CAPEX 108,900 EUR. The payback of A falls between 9 and 10 years. B pays back faster.

**Question 4.** In scenario B, most of the revenue is money the school does not spend. A third party owns the PV system on the roof. Who receives this money? Name the question the school and the third party settle in a contract.

**Question 5.** Rank A and B by simple payback. Name two things the simple payback ignores.

**Answer.**

**Solution.** Q4: the saving lands on the school bill, so the school receives it. The third party pays for the system and needs an income. The contract sets the price the school pays per PV kWh (or a rent), the length of the contract and the fate of the surplus. Q5: B first, about 5.7 years against 9.7. The simple payback ignores the time value of money, the cash after the payback year, degradation, price changes, the inverter replacement and the end of the contract. TD2 treats these points.

## Part 4. Monthly against hourly (10 min)

The monthly method pairs any kWh produced in a month with any kWh used in the same month. Apply the same rule to each of the 8,760 hours instead.

### Task 4.1. Hourly self-consumed energy

In [ ]:
hourly_mwh = np.minimum(pv, load).sum() / 1000

print(f"Monthly method: {self_consumed_mwh:.1f} MWh")
print(f"Hourly method:  {hourly_mwh:.1f} MWh")
print(f"Overestimate of the monthly method: {100 * (self_consumed_mwh / hourly_mwh - 1):.0f} %")

**Question 6.** Why does the monthly method overestimate the self-consumed energy? Give one hour of the day or one day of the week as an example.

**Answer.**

**What you carry to TP1.** The monthly and hourly self-consumed energy of the school, the year-one money of A and B, and the reason for the gap.

**Solution.** Production at noon on a Saturday, a Sunday or a holiday goes to the grid, and the demand in the early morning, the evening and at night finds no sun. Surplus and import occur in the same month. The sum of hourly minima never exceeds the minimum of the monthly sums: the monthly method gives an upper bound. School: about 107 MWh against 62.6 MWh, 70 % too high.